# Pairwise Ranking Model V2 - Evaluation & Visualization

This notebook provides comprehensive evaluation and visualization of trained ranking models with V2 features:

1. **Ranking Metrics**: Pairwise accuracy, AUC-ROC, Kendall Tau, Spearman correlation
2. **Auxiliary Task Evaluation**: MAE/RMSE for survival_rate, steps, avg_fire_damage predictions
3. **Training History**: Loss curves and metric evolution
4. **Pairwise GradCAM**: Backprop through full forward pass to capture cross-attention effects
5. **Cross-Attention Visualization**: Direct visualization of attention weights between A and B
6. **Differential GradCAM**: Compare isolated vs pairwise contexts to understand interaction effects

## 1. Setup and Imports

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm
from scipy.stats import kendalltau, spearmanr
from sklearn.metrics import roc_auc_score, mean_absolute_error, mean_squared_error

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Add project root to path
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import ranking_v2 modules
from ml.ranking_v2.config import RankingV2Config
from ml.ranking_v2.model import CrossAttentionRanker
from ml.ranking_v2.dataset import create_pairwise_dataloaders, SingleConfigDataset
from ml.ranking_v2.train import load_checkpoint
from ml.ranking_v2.evaluate import evaluate_pairwise, evaluate_per_plan_ranking, evaluate_auxiliary

print("Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Model Loading

In [ ]:
# Paths
CHECKPOINT_DIR = project_root / "checkpoints" / "ranking_v2"
MODEL_PATH = CHECKPOINT_DIR / "best_model.pt"
CONFIG_PATH = CHECKPOINT_DIR / "config.yaml"
HISTORY_PATH = CHECKPOINT_DIR / "training_history.json"

# Data directory
DATA_DIR = "combined_fast"

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Check if checkpoint exists
if not MODEL_PATH.exists():
    print(f"\nWARNING: Checkpoint not found at {MODEL_PATH}")
    print("Please train a model first using:")
    print("  python -m ml.ranking_v2.run_training --mode train --preset full")
else:
    print(f"\nFound checkpoint: {MODEL_PATH}")

In [ ]:
# Load checkpoint
print("Loading model checkpoint...")
model, checkpoint = load_checkpoint(str(MODEL_PATH), device=device)
model.eval()

# Extract config
config = RankingV2Config(**checkpoint['config'])

print(f"\nModel loaded from epoch {checkpoint['epoch'] + 1}")
print(f"Best validation AUC: {checkpoint.get('val_auc', 0):.4f}")
print(f"\nModel Configuration:")
print(f"  Latent dim: {config.latent_dim}")
print(f"  Cross-attention: {config.use_cross_attention}")
if config.use_cross_attention:
    print(f"    Attention heads: {config.attention_heads}")
    print(f"    Attention layers: {config.num_attention_layers}")
print(f"  Mining strategy: {config.mining_strategy}")
print(f"  Auxiliary tasks: {config.auxiliary_tasks}")
print(f"  Loss type: {config.loss_type}")

## 3. Load Test Data

In [ ]:
# Update config with current data paths
config.data_dir = DATA_DIR
config.floor_plans_dir = f"{DATA_DIR}/floor_plans"

# Load dataloaders
print("Loading test data...")
_, _, test_loader, stats = create_pairwise_dataloaders(config, compute_stats=True)

print(f"\nTest set size: {len(test_loader.dataset):,} pairs")

# Load single config dataset for per-plan evaluation
print("\nLoading single config dataset for ranking evaluation...")
single_config_dataset = SingleConfigDataset(
    simulation_results_file=f"{DATA_DIR}/simulation_results.jsonl",
    floor_plans_dir=config.floor_plans_dir,
    target_size=config.target_grid_size,
    max_configs=5000
)
print(f"Loaded {len(single_config_dataset):,} configurations")

## 4. Training History Visualization

In [ ]:
# Load training history
if HISTORY_PATH.exists():
    with open(HISTORY_PATH, 'r') as f:
        history = json.load(f)
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Total loss
    axes[0, 0].plot(epochs, history['train_loss'], 'o-', label='Train Loss', linewidth=2)
    axes[0, 0].plot(epochs, history['val_loss'], 's-', label='Val Loss', linewidth=2)
    best_epoch = np.argmin(history['val_loss']) + 1
    axes[0, 0].axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best (Epoch {best_epoch})')
    axes[0, 0].set_xlabel('Epoch', fontsize=12)
    axes[0, 0].set_ylabel('Total Loss', fontsize=12)
    axes[0, 0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    axes[0, 0].legend(fontsize=11)
    axes[0, 0].grid(True, alpha=0.3)
    
    # Ranking loss
    if 'train_ranking_loss' in history:
        axes[0, 1].plot(epochs, history['train_ranking_loss'], 'o-', label='Train Ranking Loss', linewidth=2, color='steelblue')
        axes[0, 1].plot(epochs, history['val_ranking_loss'], 's-', label='Val Ranking Loss', linewidth=2, color='coral')
        axes[0, 1].set_xlabel('Epoch', fontsize=12)
        axes[0, 1].set_ylabel('Ranking Loss', fontsize=12)
        axes[0, 1].set_title('Ranking Loss (RankNet/Hinge)', fontsize=14, fontweight='bold')
        axes[0, 1].legend(fontsize=11)
        axes[0, 1].grid(True, alpha=0.3)
    
    # Auxiliary loss
    if 'train_aux_loss' in history:
        axes[1, 0].plot(epochs, history['train_aux_loss'], 'o-', label='Train Aux Loss', linewidth=2, color='mediumseagreen')
        axes[1, 0].plot(epochs, history['val_aux_loss'], 's-', label='Val Aux Loss', linewidth=2, color='orange')
        axes[1, 0].set_xlabel('Epoch', fontsize=12)
        axes[1, 0].set_ylabel('Auxiliary Loss', fontsize=12)
        axes[1, 0].set_title('Auxiliary Task Loss', fontsize=14, fontweight='bold')
        axes[1, 0].legend(fontsize=11)
        axes[1, 0].grid(True, alpha=0.3)
    
    # Validation AUC
    if 'val_auc' in history:
        axes[1, 1].plot(epochs, history['val_auc'], 'o-', label='Val AUC', linewidth=2, color='mediumpurple')
        best_auc_epoch = np.argmax(history['val_auc']) + 1
        best_auc = max(history['val_auc'])
        axes[1, 1].axvline(best_auc_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best AUC (Epoch {best_auc_epoch})')
        axes[1, 1].scatter([best_auc_epoch], [best_auc], color='red', s=100, zorder=5)
        axes[1, 1].set_xlabel('Epoch', fontsize=12)
        axes[1, 1].set_ylabel('AUC-ROC', fontsize=12)
        axes[1, 1].set_title(f'Validation AUC (Best: {best_auc:.4f})', fontsize=14, fontweight='bold')
        axes[1, 1].set_ylim([0.5, 1.0])
        axes[1, 1].legend(fontsize=11)
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(CHECKPOINT_DIR / 'evaluation_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Training history not found.")
    history = None

## 5. Pairwise Ranking Evaluation

In [ ]:
# Evaluate pairwise metrics
print("Evaluating pairwise ranking metrics...")
pairwise_metrics = evaluate_pairwise(model, test_loader, device)

print("\n" + "=" * 80)
print("PAIRWISE RANKING METRICS (Test Set)")
print("=" * 80)
print(f"Pairwise Accuracy:       {pairwise_metrics['pairwise_accuracy']:.4f}")
print(f"Pairwise AUC-ROC:        {pairwise_metrics['pairwise_auc']:.4f}")
print(f"Weighted Accuracy:       {pairwise_metrics['weighted_accuracy']:.4f}")
print(f"Number of pairs:         {pairwise_metrics['num_pairs']:,}")
print("=" * 80)

## 6. Per-Plan Ranking Evaluation

In [ ]:
# Evaluate per-plan ranking metrics
print("Evaluating per-plan ranking metrics (Kendall Tau, Spearman)...")
ranking_metrics = evaluate_per_plan_ranking(model, single_config_dataset, device)

print("\n" + "=" * 80)
print("PER-PLAN RANKING METRICS")
print("=" * 80)
print(f"\nOverall Correlations:")
print(f"  Kendall Tau:           {ranking_metrics['kendall_tau']:.4f}")
print(f"  Spearman R:            {ranking_metrics['spearman_r']:.4f}")
print(f"  Pearson R:             {ranking_metrics['pearson_r']:.4f}")

print(f"\nPer-Plan Statistics:")
print(f"  Number of plans:       {ranking_metrics['num_plans']}")
print(f"  Mean Kendall Tau:      {ranking_metrics['mean_per_plan_kendall']:.4f} ± {ranking_metrics['std_per_plan_kendall']:.4f}")
print(f"  Mean Spearman R:       {ranking_metrics['mean_per_plan_spearman']:.4f} ± {ranking_metrics['std_per_plan_spearman']:.4f}")

if 'ndcg_at_10' in ranking_metrics:
    print(f"\nRanking Quality:")
    print(f"  NDCG@10:               {ranking_metrics['ndcg_at_10']:.4f}")
    print(f"  Top-10 Overlap:        {ranking_metrics.get('top_10_overlap', 0):.4f}")

print("=" * 80)

In [ ]:
# Visualize ranking metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Correlation metrics
corr_metrics = ['kendall_tau', 'spearman_r', 'pearson_r']
corr_values = [ranking_metrics[m] for m in corr_metrics]
corr_labels = ['Kendall Tau', 'Spearman R', 'Pearson R']
colors = ['steelblue', 'coral', 'mediumseagreen']

axes[0].bar(corr_labels, corr_values, color=colors, alpha=0.8)
axes[0].set_ylabel('Correlation', fontsize=12)
axes[0].set_title('Overall Ranking Correlations', fontsize=14, fontweight='bold')
axes[0].set_ylim([0, 1])
axes[0].grid(True, alpha=0.3, axis='y')

# Per-plan Kendall Tau distribution
if 'per_plan_kendall' in ranking_metrics:
    axes[1].hist(ranking_metrics['per_plan_kendall'], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
    axes[1].axvline(ranking_metrics['mean_per_plan_kendall'], color='red', linestyle='--', 
                   linewidth=2, label=f"Mean: {ranking_metrics['mean_per_plan_kendall']:.3f}")
    axes[1].set_xlabel('Kendall Tau', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title('Per-Plan Kendall Tau Distribution', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3, axis='y')

# Per-plan Spearman R distribution
if 'per_plan_spearman' in ranking_metrics:
    axes[2].hist(ranking_metrics['per_plan_spearman'], bins=30, alpha=0.7, color='coral', edgecolor='black')
    axes[2].axvline(ranking_metrics['mean_per_plan_spearman'], color='red', linestyle='--', 
                   linewidth=2, label=f"Mean: {ranking_metrics['mean_per_plan_spearman']:.3f}")
    axes[2].set_xlabel('Spearman R', fontsize=12)
    axes[2].set_ylabel('Frequency', fontsize=12)
    axes[2].set_title('Per-Plan Spearman R Distribution', fontsize=14, fontweight='bold')
    axes[2].legend(fontsize=10)
    axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'evaluation_ranking_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Auxiliary Task Evaluation

In [ ]:
# Evaluate auxiliary tasks (if enabled)
if config.auxiliary_tasks:
    print("Evaluating auxiliary task predictions...")
    aux_metrics = evaluate_auxiliary(model, single_config_dataset, device)
    
    print("\n" + "=" * 80)
    print("AUXILIARY TASK METRICS")
    print("=" * 80)
    
    for task in config.auxiliary_tasks:
        if task in aux_metrics:
            metrics = aux_metrics[task]
            print(f"\n{task}:")
            print(f"  MAE:   {metrics['mae']:.4f}")
            print(f"  RMSE:  {metrics['rmse']:.4f}")
            print(f"  R²:    {metrics['r2']:.4f}")
    
    print("\n" + "=" * 80)
else:
    print("No auxiliary tasks enabled in this model.")
    aux_metrics = None

In [ ]:
# Visualize auxiliary task metrics
if aux_metrics:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    tasks = list(aux_metrics.keys())
    mae_values = [aux_metrics[t]['mae'] for t in tasks]
    rmse_values = [aux_metrics[t]['rmse'] for t in tasks]
    r2_values = [aux_metrics[t]['r2'] for t in tasks]
    
    # MAE and RMSE
    x = np.arange(len(tasks))
    width = 0.35
    axes[0].bar(x - width/2, mae_values, width, label='MAE', alpha=0.8, color='steelblue')
    axes[0].bar(x + width/2, rmse_values, width, label='RMSE', alpha=0.8, color='coral')
    axes[0].set_ylabel('Error', fontsize=12)
    axes[0].set_title('Auxiliary Task Error Metrics', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(tasks, rotation=15)
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # R² scores
    axes[1].bar(tasks, r2_values, alpha=0.8, color='mediumseagreen')
    axes[1].set_ylabel('R² Score', fontsize=12)
    axes[1].set_title('Auxiliary Task R² Scores', fontsize=14, fontweight='bold')
    axes[1].set_ylim([0, 1])
    axes[1].tick_params(axis='x', rotation=15)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(CHECKPOINT_DIR / 'evaluation_auxiliary_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Visualization Utilities

Custom GradCAM and attention visualization functions for the ranking model.

In [ ]:
class PairwiseGradCAM:
    """
    Pairwise GradCAM: Backprop through full forward pass to capture cross-attention effects.
    
    This shows which spatial features in the floor plan are most important for the
    ranking decision when considering both A and B together.
    """
    
    def __init__(self, model: CrossAttentionRanker):
        self.model = model
        self.gradients = None
        self.activations = None
        
        # Hook into the final_conv layer of FloorPlanEncoder
        target_layer = model.floor_plan_encoder.final_conv
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)
    
    def _save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate(self, grid_a, scenario_a, grid_b, scenario_b, target='logit'):
        """
        Generate GradCAM for either config A or B in pairwise context.
        
        Args:
            grid_a, scenario_a: Configuration A
            grid_b, scenario_b: Configuration B
            target: 'logit' (default), 'score_a', or 'score_b'
        
        Returns:
            cam_a: GradCAM for config A (H, W)
            cam_b: GradCAM for config B (H, W)
        """
        self.model.eval()
        
        # Ensure batch dimension
        if grid_a.dim() == 3:
            grid_a = grid_a.unsqueeze(0)
            scenario_a = scenario_a.unsqueeze(0)
            grid_b = grid_b.unsqueeze(0)
            scenario_b = scenario_b.unsqueeze(0)
        
        grid_a = grid_a.requires_grad_(True)
        grid_b = grid_b.requires_grad_(True)
        
        # Forward pass
        outputs = self.model(grid_a, scenario_a, grid_b, scenario_b)
        
        # Select target for backprop
        if target == 'logit':
            target_output = outputs['logit'][0]
        elif target == 'score_a':
            target_output = outputs['score_a'][0]
        elif target == 'score_b':
            target_output = outputs['score_b'][0]
        else:
            raise ValueError(f"Unknown target: {target}")
        
        # Backward pass for A
        self.model.zero_grad()
        target_output.backward(retain_graph=True)
        
        # Compute GradCAM for A
        weights_a = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam_a = (weights_a * self.activations).sum(dim=1, keepdim=True)
        cam_a = F.relu(cam_a)
        cam_a = F.interpolate(cam_a, size=grid_a.shape[-2:], mode='bilinear', align_corners=False)
        cam_a = cam_a.squeeze().cpu().numpy()
        cam_a = (cam_a - cam_a.min()) / (cam_a.max() - cam_a.min() + 1e-8)
        
        # Forward pass again for B (we need fresh gradients)
        self.model.zero_grad()
        outputs = self.model(grid_a, scenario_a, grid_b, scenario_b)
        if target == 'logit':
            target_output = outputs['logit'][0]
        elif target == 'score_a':
            target_output = outputs['score_a'][0]
        elif target == 'score_b':
            target_output = outputs['score_b'][0]
        
        target_output.backward()
        
        # Compute GradCAM for B
        weights_b = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam_b = (weights_b * self.activations).sum(dim=1, keepdim=True)
        cam_b = F.relu(cam_b)
        cam_b = F.interpolate(cam_b, size=grid_b.shape[-2:], mode='bilinear', align_corners=False)
        cam_b = cam_b.squeeze().cpu().numpy()
        cam_b = (cam_b - cam_b.min()) / (cam_b.max() - cam_b.min() + 1e-8)
        
        return cam_a, cam_b


class DifferentialGradCAM:
    """
    Differential GradCAM: Compare isolated vs pairwise contexts.
    
    Shows the difference in attention when a configuration is evaluated:
    1. In isolation (without cross-attention context)
    2. In pairwise context (with cross-attention from the other config)
    """
    
    def __init__(self, model: CrossAttentionRanker):
        self.model = model
        self.pairwise_gradcam = PairwiseGradCAM(model)
    
    def generate(self, grid_a, scenario_a, grid_b, scenario_b):
        """
        Generate differential GradCAM showing isolated vs pairwise attention.
        
        Returns:
            diff_a: Difference map for config A (pairwise - isolated)
            diff_b: Difference map for config B (pairwise - isolated)
        """
        # Get pairwise GradCAM
        cam_pairwise_a, cam_pairwise_b = self.pairwise_gradcam.generate(
            grid_a, scenario_a, grid_b, scenario_b, target='logit'
        )
        
        # Get isolated GradCAM (compare A with itself, B with itself)
        cam_isolated_a, _ = self.pairwise_gradcam.generate(
            grid_a, scenario_a, grid_a, scenario_a, target='score_a'
        )
        _, cam_isolated_b = self.pairwise_gradcam.generate(
            grid_b, scenario_b, grid_b, scenario_b, target='score_b'
        )
        
        # Compute difference
        diff_a = cam_pairwise_a - cam_isolated_a
        diff_b = cam_pairwise_b - cam_isolated_b
        
        return diff_a, diff_b, cam_pairwise_a, cam_pairwise_b, cam_isolated_a, cam_isolated_b


def visualize_cross_attention_weights(model, grid_a, scenario_a, grid_b, scenario_b):
    """
    Visualize cross-attention weights between A and B.
    
    Returns attention weights from each cross-attention layer.
    """
    model.eval()
    
    # Ensure batch dimension
    if grid_a.dim() == 3:
        grid_a = grid_a.unsqueeze(0)
        scenario_a = scenario_a.unsqueeze(0)
        grid_b = grid_b.unsqueeze(0)
        scenario_b = scenario_b.unsqueeze(0)
    
    with torch.no_grad():
        # Forward pass with attention weights
        outputs = model(grid_a, scenario_a, grid_b, scenario_b)
        
        # Extract attention weights from cross-attention layers
        attention_weights = []
        if hasattr(model, 'cross_attention') and model.cross_attention is not None:
            for layer in model.cross_attention.layers:
                if hasattr(layer, 'attention_weights_a') and layer.attention_weights_a is not None:
                    # Extract attention weights (B, num_heads, 1, 1)
                    attn_a = layer.attention_weights_a[0].cpu().numpy()  # (num_heads, 1, 1)
                    attn_b = layer.attention_weights_b[0].cpu().numpy()  # (num_heads, 1, 1)
                    attention_weights.append({
                        'a_to_b': attn_a.squeeze(),  # (num_heads,)
                        'b_to_a': attn_b.squeeze()   # (num_heads,)
                    })
    
    return attention_weights, outputs


print("Visualization utilities loaded!")

## 9. Pairwise GradCAM Visualization

Backprop through the full forward pass to see which spatial features matter for ranking.

In [ ]:
# Select random pairs from test set
num_viz_samples = 3
np.random.seed(42)
viz_indices = np.random.choice(len(test_loader.dataset), size=num_viz_samples, replace=False)

print(f"Selected {num_viz_samples} random pairs for visualization")
print(f"Indices: {viz_indices.tolist()}")

In [ ]:
# Initialize GradCAM
pairwise_gradcam = PairwiseGradCAM(model)

# Visualize selected pairs
for idx in viz_indices:
    sample = test_loader.dataset[idx]
    
    grid_a = sample['grid_a'].to(device)
    scenario_a = sample['scenario_a'].to(device)
    grid_b = sample['grid_b'].to(device)
    scenario_b = sample['scenario_b'].to(device)
    label = sample['label'].item()
    
    # Generate GradCAM
    cam_a, cam_b = pairwise_gradcam.generate(grid_a, scenario_a, grid_b, scenario_b, target='logit')
    
    # Get model prediction
    with torch.no_grad():
        outputs = model(grid_a.unsqueeze(0), scenario_a.unsqueeze(0), 
                       grid_b.unsqueeze(0), scenario_b.unsqueeze(0))
        logit = outputs['logit'][0].item()
        prediction = 1 if logit > 0 else 0
    
    # Extract floor plans
    floor_plan_a = grid_a[1].cpu().numpy()  # passable channel
    floor_plan_b = grid_b[1].cpu().numpy()
    
    # Plot
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Config A
    axes[0, 0].imshow(floor_plan_a, cmap='gray')
    axes[0, 0].set_title('Config A - Floor Plan', fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(cam_a, cmap='jet')
    axes[0, 1].set_title('Config A - GradCAM', fontsize=12, fontweight='bold')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(floor_plan_a, cmap='gray')
    axes[0, 2].imshow(cam_a, cmap='jet', alpha=0.5)
    axes[0, 2].set_title('Config A - Overlay', fontsize=12, fontweight='bold')
    axes[0, 2].axis('off')
    
    # Config B
    axes[1, 0].imshow(floor_plan_b, cmap='gray')
    axes[1, 0].set_title('Config B - Floor Plan', fontsize=12, fontweight='bold')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(cam_b, cmap='jet')
    axes[1, 1].set_title('Config B - GradCAM', fontsize=12, fontweight='bold')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(floor_plan_b, cmap='gray')
    axes[1, 2].imshow(cam_b, cmap='jet', alpha=0.5)
    axes[1, 2].set_title('Config B - Overlay', fontsize=12, fontweight='bold')
    axes[1, 2].axis('off')
    
    plt.suptitle(f'Pairwise GradCAM - Sample {idx}\n'
                f'Ground Truth: {"A > B" if label == 1 else "B > A"} | '
                f'Prediction: {"A > B" if prediction == 1 else "B > A"} | '
                f'Logit: {logit:.3f}', 
                fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(CHECKPOINT_DIR / f'pairwise_gradcam_sample_{idx}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nSample {idx} - Logit: {logit:.4f}, Prediction: {'A > B' if prediction == 1 else 'B > A'}, Label: {'A > B' if label == 1 else 'B > A'}")

## 10. Cross-Attention Visualization

Direct visualization of attention weights showing how A and B attend to each other.

In [ ]:
# Only run if cross-attention is enabled
if config.use_cross_attention:
    print("Visualizing cross-attention weights...\n")
    
    for idx in viz_indices:
        sample = test_loader.dataset[idx]
        
        grid_a = sample['grid_a'].to(device)
        scenario_a = sample['scenario_a'].to(device)
        grid_b = sample['grid_b'].to(device)
        scenario_b = sample['scenario_b'].to(device)
        
        # Get attention weights
        attention_weights, outputs = visualize_cross_attention_weights(
            model, grid_a, scenario_a, grid_b, scenario_b
        )
        
        if attention_weights:
            num_layers = len(attention_weights)
            num_heads = len(attention_weights[0]['a_to_b'])
            
            fig, axes = plt.subplots(num_layers, 2, figsize=(12, 4 * num_layers))
            if num_layers == 1:
                axes = axes.reshape(1, -1)
            
            for layer_idx, attn in enumerate(attention_weights):
                # A attends to B
                axes[layer_idx, 0].bar(range(num_heads), attn['a_to_b'], alpha=0.8, color='steelblue')
                axes[layer_idx, 0].set_xlabel('Attention Head', fontsize=11)
                axes[layer_idx, 0].set_ylabel('Attention Weight', fontsize=11)
                axes[layer_idx, 0].set_title(f'Layer {layer_idx + 1}: A attends to B', fontsize=12, fontweight='bold')
                axes[layer_idx, 0].set_ylim([0, 1])
                axes[layer_idx, 0].grid(True, alpha=0.3, axis='y')
                
                # B attends to A
                axes[layer_idx, 1].bar(range(num_heads), attn['b_to_a'], alpha=0.8, color='coral')
                axes[layer_idx, 1].set_xlabel('Attention Head', fontsize=11)
                axes[layer_idx, 1].set_ylabel('Attention Weight', fontsize=11)
                axes[layer_idx, 1].set_title(f'Layer {layer_idx + 1}: B attends to A', fontsize=12, fontweight='bold')
                axes[layer_idx, 1].set_ylim([0, 1])
                axes[layer_idx, 1].grid(True, alpha=0.3, axis='y')
            
            plt.suptitle(f'Cross-Attention Weights - Sample {idx}', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig(CHECKPOINT_DIR / f'cross_attention_sample_{idx}.png', dpi=150, bbox_inches='tight')
            plt.show()
        else:
            print(f"No attention weights found for sample {idx}")
else:
    print("Cross-attention is not enabled in this model.")

## 11. Differential GradCAM

Compare isolated vs pairwise contexts to understand interaction effects.

In [ ]:
# Initialize Differential GradCAM
diff_gradcam = DifferentialGradCAM(model)

print("Generating Differential GradCAM visualizations...\n")

for idx in viz_indices:
    sample = test_loader.dataset[idx]
    
    grid_a = sample['grid_a'].to(device)
    scenario_a = sample['scenario_a'].to(device)
    grid_b = sample['grid_b'].to(device)
    scenario_b = sample['scenario_b'].to(device)
    
    # Generate differential GradCAM
    diff_a, diff_b, cam_pair_a, cam_pair_b, cam_iso_a, cam_iso_b = diff_gradcam.generate(
        grid_a, scenario_a, grid_b, scenario_b
    )
    
    # Extract floor plans
    floor_plan_a = grid_a[1].cpu().numpy()
    floor_plan_b = grid_b[1].cpu().numpy()
    
    # Plot
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    # Config A
    axes[0, 0].imshow(floor_plan_a, cmap='gray')
    axes[0, 0].set_title('Config A - Floor Plan', fontsize=11, fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(cam_iso_a, cmap='jet')
    axes[0, 1].set_title('Isolated\n(No Context)', fontsize=11, fontweight='bold')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(cam_pair_a, cmap='jet')
    axes[0, 2].set_title('Pairwise\n(With B Context)', fontsize=11, fontweight='bold')
    axes[0, 2].axis('off')
    
    vmax_a = max(abs(diff_a.min()), abs(diff_a.max()))
    im_a = axes[0, 3].imshow(diff_a, cmap='RdBu_r', vmin=-vmax_a, vmax=vmax_a)
    axes[0, 3].set_title('Difference\n(Pairwise - Isolated)', fontsize=11, fontweight='bold')
    axes[0, 3].axis('off')
    plt.colorbar(im_a, ax=axes[0, 3], fraction=0.046)
    
    # Config B
    axes[1, 0].imshow(floor_plan_b, cmap='gray')
    axes[1, 0].set_title('Config B - Floor Plan', fontsize=11, fontweight='bold')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(cam_iso_b, cmap='jet')
    axes[1, 1].set_title('Isolated\n(No Context)', fontsize=11, fontweight='bold')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(cam_pair_b, cmap='jet')
    axes[1, 2].set_title('Pairwise\n(With A Context)', fontsize=11, fontweight='bold')
    axes[1, 2].axis('off')
    
    vmax_b = max(abs(diff_b.min()), abs(diff_b.max()))
    im_b = axes[1, 3].imshow(diff_b, cmap='RdBu_r', vmin=-vmax_b, vmax=vmax_b)
    axes[1, 3].set_title('Difference\n(Pairwise - Isolated)', fontsize=11, fontweight='bold')
    axes[1, 3].axis('off')
    plt.colorbar(im_b, ax=axes[1, 3], fraction=0.046)
    
    plt.suptitle(f'Differential GradCAM - Sample {idx}\n'
                'Red: More attention in pairwise | Blue: Less attention in pairwise',
                fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(CHECKPOINT_DIR / f'differential_gradcam_sample_{idx}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Sample {idx}: Difference range A=[{diff_a.min():.3f}, {diff_a.max():.3f}], B=[{diff_b.min():.3f}, {diff_b.max():.3f}]\n")

## 12. Summary Report

In [ ]:
# Create comprehensive summary
print("=" * 80)
print("COMPREHENSIVE EVALUATION SUMMARY")
print("=" * 80)

print(f"\nModel: {MODEL_PATH}")
print(f"Configuration:")
print(f"  Latent dim: {config.latent_dim}")
print(f"  Cross-attention: {config.use_cross_attention}")
print(f"  Mining strategy: {config.mining_strategy}")
print(f"  Auxiliary tasks: {config.auxiliary_tasks}")

print("\n" + "-" * 80)
print("PAIRWISE RANKING METRICS")
print("-" * 80)
print(f"Pairwise Accuracy:  {pairwise_metrics['pairwise_accuracy']:.4f}")
print(f"Pairwise AUC-ROC:   {pairwise_metrics['pairwise_auc']:.4f}")

print("\n" + "-" * 80)
print("PER-PLAN RANKING METRICS")
print("-" * 80)
print(f"Kendall Tau:        {ranking_metrics['kendall_tau']:.4f}")
print(f"Spearman R:         {ranking_metrics['spearman_r']:.4f}")
print(f"Pearson R:          {ranking_metrics['pearson_r']:.4f}")

if aux_metrics:
    print("\n" + "-" * 80)
    print("AUXILIARY TASK METRICS")
    print("-" * 80)
    for task in config.auxiliary_tasks:
        if task in aux_metrics:
            print(f"{task}: MAE={aux_metrics[task]['mae']:.4f}, RMSE={aux_metrics[task]['rmse']:.4f}, R²={aux_metrics[task]['r2']:.4f}")

print("\n" + "=" * 80)
print("VISUALIZATIONS GENERATED")
print("=" * 80)
print("- Training history: evaluation_training_history.png")
print("- Ranking metrics: evaluation_ranking_metrics.png")
if aux_metrics:
    print("- Auxiliary metrics: evaluation_auxiliary_metrics.png")
print("- Pairwise GradCAM: pairwise_gradcam_sample_*.png")
if config.use_cross_attention:
    print("- Cross-attention: cross_attention_sample_*.png")
print("- Differential GradCAM: differential_gradcam_sample_*.png")
print(f"\nAll files saved to: {CHECKPOINT_DIR}")
print("=" * 80)

## 13. Save Evaluation Results

In [ ]:
# Save comprehensive evaluation results
evaluation_results = {
    'model_path': str(MODEL_PATH),
    'config': {
        'latent_dim': config.latent_dim,
        'use_cross_attention': config.use_cross_attention,
        'attention_heads': config.attention_heads if config.use_cross_attention else None,
        'num_attention_layers': config.num_attention_layers if config.use_cross_attention else None,
        'mining_strategy': config.mining_strategy,
        'auxiliary_tasks': config.auxiliary_tasks,
        'loss_type': config.loss_type
    },
    'pairwise_metrics': pairwise_metrics,
    'ranking_metrics': {
        'kendall_tau': ranking_metrics['kendall_tau'],
        'spearman_r': ranking_metrics['spearman_r'],
        'pearson_r': ranking_metrics['pearson_r'],
        'num_plans': ranking_metrics['num_plans'],
        'mean_per_plan_kendall': ranking_metrics['mean_per_plan_kendall'],
        'mean_per_plan_spearman': ranking_metrics['mean_per_plan_spearman']
    }
}

if aux_metrics:
    evaluation_results['auxiliary_metrics'] = aux_metrics

# Save to file
eval_results_path = CHECKPOINT_DIR / 'evaluation_results.json'
with open(eval_results_path, 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print(f"Evaluation results saved to: {eval_results_path}")

## Done!

This notebook has completed a comprehensive evaluation of your Pairwise Ranking Model V2.

Key insights from the visualizations:
- **Pairwise GradCAM**: Shows which spatial features are important for the ranking decision
- **Cross-Attention**: Reveals how configurations attend to each other's features
- **Differential GradCAM**: Highlights the effect of cross-attention by comparing isolated vs pairwise contexts

All results and visualizations have been saved to the checkpoints directory.